# M3.6 · BatchNorm, LayerNorm, RMSNorm

Aktivasyonları her adımda yeniden ölçeklemek. Üç örnekte BatchNorm'un ileri ve (meşhur) geri geçişini elle hesaplıyor, saf Python'da yazıyor, NumPy'da CS231n'in train/test modlu BatchNorm'unu ve LayerNorm'unu kuruyor, PyTorch'ta `nn.BatchNorm1d` ve `nn.RMSNorm` ile bit düzeyinde eşleşen kendi modüllerini yazıyor ve en sonda BN/LN/IN/GN'yi tek bir "hangi eksenlerde normalize?" fonksiyonuna indiriyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m3-6-norm.html](https://m-gonul.github.io/pytorch-lab/m3-6-norm.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m3_6_norm.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def hand():\n        out, cache = need(ns, \"bn_forward\")([[1.0], [2.0], [3.0]], [2.0], [1.0], eps=0.0)\n        close([r[0] for r in out], [1 - 2 * 1.5 ** 0.5, 1, 1 + 2 * 1.5 ** 0.5])\n        dX, dg, db = need(ns, \"bn_backward\")([[1.0], [0.0], [0.0]], cache)\n        close([r[0] for r in dX], [0.408248, -0.816497, 0.408248], atol=1e-6); close(dg, [-1.224745], atol=1e-6); close(db, [1])\n    R.test(\"elle örnek\", hand)\n\n    def num():\n        rng = random.Random(0)\n        X = [[rng.gauss(0, 2) for _ in range(3)] for _ in range(5)]\n        g = [rng.uniform(0.5, 2) for _ in range(3)]; b = [rng.gauss(0, 1) for _ in range(3)]\n        W = [[rng.gauss(0, 1) for _ in range(3)] for _ in range(5)]\n        L = lambda X_: sum(o * w for ro, rw in zip(need(ns, \"bn_forward\")(X_, g, b)[0], W) for o, w in zip(ro, rw))\n        _, cache = need(ns, \"bn_forward\")(X, g, b)\n        dX, _, _ = need(ns, \"bn_backward\")(W, cache)\n        h = 1e-6\n        for i in range(5):\n            for j in range(3):\n                Xp = [r[:] for r in X]; Xm = [r[:] for r in X]; Xp[i][j] += h; Xm[i][j] -= h\n                close(dX[i][j], (L(Xp) - L(Xm)) / (2 * h), rtol=1e-4, atol=1e-6, what=f\"dX[{i}][{j}]\")\n        ensure(all(abs(sum(dX[i][j] for i in range(5))) < 1e-9 for j in range(3)), \"Σᵢ dXᵢ sıfır olmalı\")\n    R.test(\"bn_backward sayısal gradyanla aynı, sütun toplamları 0\", num)\n    R.test(\"rms_norm\", lambda: close(need(ns, \"rms_norm\")([3.0, 4.0], [1.0, 2.0], eps=0.0), [3 / 12.5 ** 0.5, 8 / 12.5 ** 0.5]))\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef num(f, x, dout, h=1e-6):\n    g = np.zeros_like(x)\n    for idx in np.ndindex(x.shape):\n        old = x[idx]; x[idx] = old + h; p = (f(x) * dout).sum(); x[idx] = old - h; m = (f(x) * dout).sum(); x[idx] = old\n        g[idx] = (p - m) / (2 * h)\n    return g\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    x = rng.normal(3, 2, size=(8, 4)); gamma, beta = rng.uniform(0.5, 2, 4), rng.normal(size=4); dout = rng.normal(size=(8, 4))\n\n    def bnf():\n        p = {\"mode\": \"train\"}\n        out, _ = need(ns, \"batchnorm_forward\")(x, gamma, beta, p)\n        close(out.mean(0), beta, atol=1e-9, what=\"çıktı ortalaması β\"); close(out.std(0), gamma * np.sqrt(x.var(0) / (x.var(0) + 1e-5)), rtol=1e-6, what=\"çıktı std'si ≈ γ\")\n        close(p[\"running_mean\"], 0.1 * x.mean(0), what=\"running_mean (momentum 0.9)\"); close(p[\"running_var\"], 0.9 + 0.1 * x.var(0), what=\"running_var\")\n        p[\"mode\"] = \"test\"\n        out_t, _ = need(ns, \"batchnorm_forward\")(x, gamma, beta, p)\n        close(out_t, gamma * (x - p[\"running_mean\"]) / np.sqrt(p[\"running_var\"] + 1e-5) + beta, what=\"test modu\")\n    R.test(\"batchnorm_forward: train/test ve running istatistikler\", bnf)\n\n    def bnb():\n        f = lambda xx: need(ns, \"batchnorm_forward\")(xx, gamma, beta, {\"mode\": \"train\"})[0]\n        _, cache = need(ns, \"batchnorm_forward\")(x, gamma, beta, {\"mode\": \"train\"})\n        dx, dg, db = need(ns, \"batchnorm_backward\")(dout, cache)\n        close(dx, num(f, x.copy(), dout), rtol=1e-5, atol=1e-7, what=\"dx\")\n        close(db, dout.sum(0), what=\"dβ\")\n    R.test(\"batchnorm_backward sayısal gradyanla aynı\", bnb)\n\n    def ln():\n        g2, b2 = rng.uniform(0.5, 2, 4), rng.normal(size=4)\n        out, cache = need(ns, \"layernorm_forward\")(x, g2, b2)\n        close(((out - b2) / g2).mean(1), np.zeros(8), atol=1e-9, what=\"satır ortalaması\")\n        dx, dg, db = need(ns, \"layernorm_backward\")(dout, cache)\n        close(dx, num(lambda xx: need(ns, \"layernorm_forward\")(xx, g2, b2)[0], x.copy(), dout), rtol=1e-5, atol=1e-7, what=\"dx\")\n        xh = (x - x.mean(1, keepdims=True)) / np.sqrt(x.var(1, keepdims=True) + 1e-5)\n        close(dg, (dout * xh).sum(0), what=\"dγ\")\n    R.test(\"layernorm forward/backward\", ln)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n\n    def bn():\n        C = need(ns, \"MyBatchNorm1d\", \"sınıf\")\n        a, b = C(6).double(), nn.BatchNorm1d(6).double()\n        with torch.no_grad():\n            w, bb = torch.rand(6, generator=g, dtype=torch.float64) + 0.5, torch.randn(6, generator=g, dtype=torch.float64)\n            a.weight.copy_(w); a.bias.copy_(bb); b.weight.copy_(w); b.bias.copy_(bb)\n        ensure(set(dict(a.named_buffers())) >= {\"running_mean\", \"running_var\"}, \"running_mean/var buffer olarak kayıtlı olmalı\")\n        for _ in range(5):\n            x = torch.randn(16, 6, generator=g, dtype=torch.float64) * 3 + 1\n            xa, xb = x.clone().requires_grad_(True), x.clone().requires_grad_(True)\n            ya, yb = a(xa), b(xb)\n            close(ya, yb, what=\"eğitim çıktısı\")\n            w = torch.randn(16, 6, generator=g, dtype=torch.float64)\n            (ya * w).sum().backward(); (yb * w).sum().backward()\n            close(xa.grad, xb.grad, what=\"dx\")\n        close(a.running_mean, b.running_mean, what=\"running_mean\"); close(a.running_var, b.running_var, what=\"running_var (yansız!)\")\n        a.eval(); b.eval()\n        x = torch.randn(4, 6, generator=g, dtype=torch.float64)\n        close(a(x), b(x), what=\"eval çıktısı\")\n        close(a.weight.grad, b.weight.grad, what=\"dγ\")\n    R.test(\"MyBatchNorm1d = nn.BatchNorm1d (5 adım, train + eval)\", bn)\n\n    def rms():\n        C = need(ns, \"MyRMSNorm\", \"sınıf\")\n        a, b = C(8, eps=1e-6).double(), nn.RMSNorm(8, eps=1e-6).double()\n        with torch.no_grad():\n            w = torch.rand(8, generator=g, dtype=torch.float64) + 0.5\n            a.weight.copy_(w); b.weight.copy_(w)\n        x = torch.randn(3, 5, 8, generator=g, dtype=torch.float64)\n        xa, xb = x.clone().requires_grad_(True), x.clone().requires_grad_(True)\n        ya, yb = a(xa), b(xb)\n        close(ya, yb)\n        ya.sum().backward(); yb.sum().backward()\n        close(xa.grad, xb.grad, what=\"dx\")\n    R.test(\"MyRMSNorm = nn.RMSNorm\", rms)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    x = torch.randn(3, 8, 5, 5, generator=g, dtype=torch.float64) * 2 + 1\n    gm, bt = torch.rand(8, generator=g, dtype=torch.float64) + 0.5, torch.randn(8, generator=g, dtype=torch.float64)\n    gn = lambda *a: need(ns, \"group_norm\")(*a)\n    R.test(\"group_norm = F.group_norm (G = 4)\", lambda: close(gn(x, 4, gm, bt), F.group_norm(x, 4, gm, bt)))\n    R.test(\"G = C → instance_norm\", lambda: close(gn(x, 8, torch.ones(8, dtype=x.dtype), torch.zeros(8, dtype=x.dtype)), F.instance_norm(x)))\n    R.test(\"G = 1 → layer_norm (C, H, W)\", lambda: close(gn(x, 1, torch.ones(8, dtype=x.dtype), torch.zeros(8, dtype=x.dtype)), F.layer_norm(x, x.shape[1:])))\n    R.test(\"norm_over(x, (0, 2, 3)) → batch_norm (eğitim)\", lambda: close(need(ns, \"norm_over\")(x, (0, 2, 3)), F.batch_norm(x, None, None, training=True)))\n    if \"group_norm\" not in ns:\n        return\n    xb = torch.randn(32, 64, 16, 16, generator=g)\n    o, z = torch.ones(64), torch.zeros(64)\n\n    def loop():\n        out = torch.empty_like(xb)\n        for k in range(8):\n            s = slice(8 * k, 8 * k + 8)\n            blk = xb[:, s]\n            mu = blk.mean((1, 2, 3), keepdim=True); var = blk.var((1, 2, 3), keepdim=True, unbiased=False)\n            out[:, s] = (blk - mu) / torch.sqrt(var + 1e-5)\n        return out\n    t_loop = bench(loop, repeat=3)\n    t_vec = bench(lambda: ns[\"group_norm\"](xb, 8, o, z), repeat=3)\n    t_ref = bench(lambda: F.group_norm(xb, 8, o, z), repeat=3)\n    R.table(\"GroupNorm, (32, 64, 16, 16), G = 8\", [\"yöntem\", \"süre\"],\n            [[\"for grup: dilimle + normalize\", fmt_time(t_loop)], [\"senin group_norm (reshape)\", fmt_time(t_vec)], [\"F.group_norm (fused kernel)\", fmt_time(t_ref)]],\n            note=\"Fused kernel ara tensörleri (mean, var, x − μ) belleğe yazmadan tek geçişte hesaplar; M6.5/M7.2'de bunu kendin yapacaksın.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m3_6_norm as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Normalizasyon katmanları aktivasyonları sıfır ortalama, birim varyansa çeker, sonra öğrenilen bir ölçek $\gamma$ ve kaydırma $\beta$ uygular:

$$
\hat x=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}},\qquad y=\gamma\odot\hat x+\beta .
$$

Hepsinin farkı $\mu$ ve $\sigma^2$'nin **hangi eksenler** üzerinden hesaplandığı ($N$ batch, $C$ kanal/özellik, $H\times W$ uzamsal):

| katman | istatistik eksenleri | $\gamma,\beta$ boyutu |
|---|---|---|
| BatchNorm | $N\,(,H,W)$ | $C$ |
| LayerNorm | $C\,(,H,W)$ | $C$ (ya da $C,H,W$) |
| InstanceNorm | $H,W$ | $C$ |
| GroupNorm | $C/G$ grubu $,H,W$ | $C$ |
| RMSNorm | $C$; ortalama çıkarılmaz, $\sqrt{\text{mean}(x^2)}$'e bölünür | $C$ |

BatchNorm eğitimde batch istatistiğini, değerlendirmede eğitim boyunca biriktirilen **running** istatistikleri kullanır. Geri geçişi, $\mu$ ve $\sigma$'nın tüm batch'e bağlı olması yüzünden üç terimlidir:

$$
dx=\frac{1}{N\sigma}\Big(N\,d\hat x-\sum_j d\hat x_j-\hat x\sum_j d\hat x_j\hat x_j\Big),\quad d\hat x=\gamma\,dy,\quad d\gamma=\sum dy\,\hat x,\quad d\beta=\sum dy .
$$

## 1 · Elle

Üç örneği normalize et; sonra yalnız ilk çıktıya gradyan gönder ($dy=[1,0,0]$). Gradyan yalnız $x_1$'e değil, **üç** girdiye de dağılıyor ve toplamı sıfır: $0.408-0.816+0.408=0$. Tüm girdileri aynı miktarda kaydırmak $\hat x$'yi değiştirmez; gradyan da bu yönde sıfır.

RMSNorm'da ortalama çıkarılmıyor; yalnız ölçek normalize ediliyor. Daha ucuz ve LLaMA gibi modellerde LayerNorm kadar iyi çalışıyor.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**BatchNorm, tek özellik, batch $x=[1,2,3]$, $\epsilon=0$, $\gamma=2$, $\beta=1$**

- `mu`: $\mu$
- `var`: $\sigma^2$ (MLE, $\frac1N$)
- `xh1`: $\hat x_1=(x_1-\mu)/\sigma$
- `y1`: $y_1=\gamma\hat x_1+\beta$

**Geri geçiş, $\partial L/\partial y=[1,0,0]$: $dx=\frac{1}{N\sigma}\big(N\,d\hat x-\sum d\hat x-\hat x\sum d\hat x\,\hat x\big)$, $d\hat x=\gamma\,dy$**

- `dx1`: $\partial L/\partial x_1$
- `dx2`: $\partial L/\partial x_2$
- `dg`: $\partial L/\partial\gamma=\sum dy\,\hat x$

**RMSNorm: $x=[3,4]$, $g=1$, $\epsilon=0$: $y=x/\sqrt{\text{mean}(x^2)}$**

- `rms1`: $y_1$

In [ ]:
cevaplar = dict(
    mu=None,
    var=None,
    xh1=None,
    y1=None,
    dx1=None,
    dx2=None,
    dg=None,
    rms1=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"mu":[2,0],"var":[0.666667,0.001],"xh1":[-1.224745,0.002],"y1":[-1.44949,0.003],"dx1":[0.408248,0.002],"dx2":[-0.816497,0.002],"dg":[-1.224745,0.002],"rms1":[0.848528,0.002]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. `X` liste listesi $(N\times D)$, istatistikler sütun başına.

- `bn_forward(X, gamma, beta, eps=1e-5)` → `(out, cache)`
- `bn_backward(dout, cache)` → `(dX, dgamma, dbeta)`; intro'daki sadeleşmiş formül.
- `rms_norm(x, g, eps=1e-6)` — tek vektör.

In [ ]:
%%writefile stage_b.py
import math


def bn_forward(X, gamma, beta, eps=1e-5):
    ...


def bn_backward(dout, cache):
    ...


def rms_norm(x, g, eps=1e-6):
    ...


out, cache = bn_forward([[1.0], [2.0], [3.0]], [2.0], [1.0], eps=0.0)
print(out)                                   # [[-1.449], [1.0], [3.449]]
print(bn_backward([[1.0], [0.0], [0.0]], cache))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

CS231n arayüzü, NumPy, **döngü yok**:

- `batchnorm_forward(x, gamma, beta, bn_param)` → `(out, cache)`. `bn_param`: `mode` (`"train"`/`"test"`), `eps` (varsayılan 1e-5), `momentum` (varsayılan 0.9), `running_mean`, `running_var` (yoksa sıfır/bir ile oluştur). Eğitimde $\text{running}=m\cdot\text{running}+(1-m)\cdot\text{batch}$ (CS231n konvansiyonu, yanlı varyans) ve `bn_param`'ı **yerinde** güncelle; testte running istatistiklerle normalize et.
- `batchnorm_backward(dout, cache)` → `(dx, dgamma, dbeta)`
- `layernorm_forward(x, gamma, beta, eps=1e-5)` → `(out, cache)`; istatistik her **satırda**.
- `layernorm_backward(dout, cache)` → `(dx, dgamma, dbeta)`

In [ ]:
%%writefile stage_c.py
import numpy as np


def batchnorm_forward(x, gamma, beta, bn_param):
    ...


def batchnorm_backward(dout, cache):
    ...


def layernorm_forward(x, gamma, beta, eps=1e-5):
    ...


def layernorm_backward(dout, cache):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch modülleri, öğrenilen parametreler `nn.Parameter`, öğrenilmeyen durum `register_buffer`:

- `MyBatchNorm1d(num_features, eps=1e-5, momentum=0.1)` — `nn.BatchNorm1d` ile aynı: eğitimde batch istatistiği (yanlı varyansla normalize), running güncellemesi $\text{run}=(1-m)\,\text{run}+m\cdot\text{batch}$, running_var için **yansız** batch varyansı; `eval()`'de running. `forward` yalnız `(N, C)` girdi.
- `MyRMSNorm(dim, eps=1e-6)` — `nn.RMSNorm(dim, eps=eps)` ile aynı: $y=x/\sqrt{\text{mean}(x^2)+\epsilon}\odot w$, son eksende.

Kontrol ileri çıktıları, gradyanları ve 5 eğitim adımı sonrası running istatistikleri karşılaştırır. Autograd'ı kullanabilirsin; elle backward gerekmiyor.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn


class MyBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        ...

    def forward(self, x):
        ...


class MyRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        ...

    def forward(self, x):
        ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Tablodaki tüm normalizasyonlar tek bir fonksiyon: "şu eksenlerde ortalama ve varyans al".

- `norm_over(x, dims, eps=1e-5)` — `x`'i `dims` eksenlerinde normalize et (yanlı varyans), $\gamma/\beta$ yok.
- `group_norm(x, G, gamma, beta, eps=1e-5)` — `x: (N, C, H, W)`; `(N, G, C//G, H, W)`'ye reshape, `norm_over` ile `(2, 3, 4)` eksenlerinde normalize, geri reshape, kanal başına $\gamma,\beta$. **Döngü yok.**

Kontrol: `group_norm` = `F.group_norm`; `G=C` → `F.instance_norm`; `G=1` → `F.layer_norm` (C, H, W üzerinde); `norm_over(x, (0, 2, 3))` → BatchNorm'un eğitim normalizasyonu. Hız tablosu: gruplar üzerinde döngü vs reshape.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def norm_over(x, dims, eps=1e-5):
    ...


def group_norm(x, G, gamma, beta, eps=1e-5):
    """x: (N, C, H, W). Döngü yok."""
    ...


x = torch.randn(2, 6, 4, 4)
print((group_norm(x, 3, torch.ones(6), torch.zeros(6)) - F.group_norm(x, 3)).abs().max())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [Ioffe & Szegedy — Batch Normalization](https://arxiv.org/abs/1502.03167) — arXiv
- [CS231n Assignment 2 — BatchNorm / LayerNorm (forward, backward, backward_alt)](https://cs231n.github.io/assignments2025/assignment2/) — MIT
- [Ba, Kiros, Hinton — Layer Normalization; Zhang & Sennrich — RMSNorm](https://arxiv.org/abs/1910.07467) — arXiv
- [Wu & He — Group Normalization](https://arxiv.org/abs/1803.08494) — arXiv
- [Stanford CS336 — Lecture 3: Architectures (pre-norm, RMSNorm)](https://stanford-cs336.github.io/) — doğrulanmadı